In [40]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer


In [41]:
from datasets import load_dataset

dataset = load_dataset("Salesforce/wikitext", "wikitext-2-v1")

In [42]:
print(dataset)

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})


In [43]:
df = pd.DataFrame(dataset['train'])

In [44]:
df = df.head(2000)

In [45]:
df.isnull().sum()

,0
text,0


In [46]:
print(f"Total rows: {len(df)}")

Total rows: 2000


In [47]:
df = df[df['text'].str.strip() != '']

In [48]:
print(f"Total rows: {len(df)}")

Total rows: 1290


In [49]:
tokenizer = Tokenizer()

In [50]:
tokenizer.fit_on_texts(df['text'])

In [51]:
len(tokenizer.word_index)

10042

In [52]:
df['tokenized'] = tokenizer.texts_to_sequences(df['text'])
print(df[['text', 'tokenized']].head(10))

                                                 text  \
1                      = Valkyria Chronicles III = \n   
3    Senjō no Valkyria 3 : <unk> Chronicles ( Japa...   
4    The game began development in 2010 , carrying...   
5    It met with positive sales in Japan , and was...   
7                                 = = Gameplay = = \n   
9    As with previous <unk> Chronicles games , Val...   
10   The game 's battle system , the <unk> system ...   
11   Troops are divided into five classes : Scouts...   
13                                    = = Plot = = \n   
15   The game takes place during the Second Europa...   

                                            tokenized  
1                                     [178, 280, 683]  
3   [2453, 68, 178, 137, 3, 280, 1836, 3585, 2454,...  
4   [1, 48, 139, 383, 5, 281, 2113, 70, 7, 292, 21...  
5   [22, 1326, 9, 902, 2463, 5, 2112, 4, 8, 828, 1...  
7                                              [1215]  
9   [11, 9, 455, 3, 280, 131, 178, 2

In [53]:
SEQ_LEN = 5

all_tokens = []
for seq in tokenizer.texts_to_sequences(df['text']):
    all_tokens.extend(seq)

X, y = [], []
for i in range(SEQ_LEN, len(all_tokens)):
    X.append(all_tokens[i-SEQ_LEN:i])
    y.append(all_tokens[i])

X = np.array(X)
y = np.array(y)

print(f"X shape: {X.shape}")  # should be (N, 5)
print(f"y shape: {y.shape}")


X shape: (100406, 5)
y shape: (100406,)


In [61]:
X.shape

(100406, 5)

In [62]:
y.shape

(100406,)

In [66]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [67]:
model = Sequential()
model.add(Embedding(10043, 100, input_length=max_len-1))
model.add(LSTM(150))
model.add(Dense(10043, activation='softmax'))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [68]:
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [69]:
import tensorflow as tf
tf.keras.backend.clear_session()

In [72]:
model.fit(X, y, epochs=50, batch_size=256, verbose=1)

Epoch 1/50
393/393 ━━━━━━━━━━━━━━━━━━━━ 73s 185ms/step - accuracy: 0.8769 - loss: 0.7107
Epoch 2/50
393/393 ━━━━━━━━━━━━━━━━━━━━ 68s 174ms/step - accuracy: 0.8832 - loss: 0.6765
Epoch 3/50
393/393 ━━━━━━━━━━━━━━━━━━━━ 67s 170ms/step - accuracy: 0.8895 - loss: 0.6446
Epoch 4/50
393/393 ━━━━━━━━━━━━━━━━━━━━ 67s 170ms/step - accuracy: 0.8961 - loss: 0.6125
Epoch 5/50
393/393 ━━━━━━━━━━━━━━━━━━━━ 67s 170ms/step - accuracy: 0.9008 - loss: 0.5827
Epoch 6/50
393/393 ━━━━━━━━━━━━━━━━━━━━ 82s 170ms/step - accuracy: 0.9080 - loss: 0.5535
Epoch 7/50
393/393 ━━━━━━━━━━━━━━━━━━━━ 82s 170ms/step - accuracy: 0.9119 - loss: 0.5254
Epoch 8/50
393/393 ━━━━━━━━━━━━━━━━━━━━ 82s 169ms/step - accuracy: 0.9173 - loss: 0.4999
Epoch 9/50
393/393 ━━━━━━━━━━━━━━━━━━━━ 82s 170ms/step - accuracy: 0.9219 - loss: 0.4744
Epoch 10/50
393/393 ━━━━━━━━━━━━━━━━━━━━ 67s 169ms/step - accuracy: 0.9263 - loss: 0.4511
Epoch 11/50
393/393 ━━━━━━━━━━━━━━━━━━━━ 82s 168ms/step - accuracy: 0.9302 - loss: 0.4291
Epoch 12/50
393/393

In [ ]:
from google.colab import files

# save first
model.save('next_word_model_new.keras')

import pickle
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

# download to your computer
files.download('next_word_model_new.keras')
files.download('tokenizer.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>